In [ ]:
import torch
import kan

# Initialize model if it's not defined (e.g., if previous cells were not run)
# For a fully trained model, ensure cell QcjXvnKota6A (where model is trained and pruned)
# is executed before this cell.
# This assumes 'kan', 'SEED' are defined from previous setup cells.
model = kan.KAN(width=[3, 5, 1], grid=3, k=3, seed=SEED)

with torch.no_grad():
  y_spline_s = model(X_test)

r2_spline = r2_score(y_test.numpy(), y_spline_s.numpy())
print(f"Spline KAN  R^2 = {r2_spline:.4f}")

# Generate symbolic formula
model.auto_symbolic()
sym_expr = kan.utils.ex_round(model.symbolic_formula()[0][0], 4)

print(f"\n  T_scaled = {sym_expr}\n")
print("Note: variables in this quation are min-max scaled to [0, 1].")

with torch.no_grad():
  y_sym_s = model(X_test) # Corrected x_test to X_test

r2_symbolic = r2_score(y_test.numpy(), y_sym_s.numpy())
print(f"Symbolic KAN  R^2 = {r2_symbolic:.4f}") # Corrected to print r2_symbolic

In [ ]:
y_test_np = scaler_y.inverse_transform(y_test.numpy())
y_spline_s = scaler_y.inverse_transform(y_spline_s)
y_sym_s = scaler_y.inverse_transform(y_sym_s)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

ax = axes[0]
lo, hi = y_test_np.min() - 10, y_test_np.max() + 10
ax.scatter(y_test_np, y_spline_s, alpha=0.5, s=14,
           label=f"Spline  R^2{r2_spline:.4f}")
ax.scatter(y_test_np, y_sym_s, alpha=0.5, s=14,
           label=f"Symbolic  R^2{r2_symbolic:.4f}")
ax.plot([lo, hi], [lo,hi], "k--")
ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
ax.set_xlabel("True T[K]"); ax.set_ylabel("Predicted T [K]")
ax.set_title("Parity plot");ax.legend()

ax = axes[1]
residuals = y_sym_s - y_test_np # Assuming y_sym_s and y_test_np are the correct variables for residuals
lo, hi = residuals.min(), residuals.max()
ax.hist(residuals, bins=40, edgecolor="white", linewidth=0.5, color="steelblue", label="Residuals") # Added label for legend
ax.axvline(0, color="k", linestyle="--")
ax.set_xlabel("Residual (Predicted - True) [K]")
ax.set_ylabel("Count")
ax.set_title(f"Symbolic KAN residuals (mean = {residuals.mean():.2f})");ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
y_test_np = scaler_y.inverse_transform(y_test.numpy())
y_spline_s = scaler_y.inverse_transform(y_spline_s)
y_sym_s = scaler_y.inverse_transform(y_sym_s)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

ax = axes[0]
lo, hi = y_test_np.min() - 10, y_test_np.max() + 10
ax.scatter(y_test_np, y_spline_s, alpha=0.5, s=14,
           label=f"Spline  R^2{r2_spline:.4f}")
ax.scatter(y_test_np, y_sym_s, alpha=0.5, s=14,
           label=f"Symbolic  R^2{r2_symbolic:.4f}")
ax.plot([lo, hi], [lo,hi], "k--")
ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
ax.set_xlabel("True T[K]"); ax.set_ylabel("Predicted T [K]")
ax.set_title("Parity plot");ax.legend()

ax = axes[1]
residuals = y_sym_s - y_test_np # Assuming y_sym_s and y_test_np are the correct variables for residuals
lo, hi = residuals.min(), residuals.max()
ax.hist(residuals, bins=40, edgecolor="white", linewidth=0.5, color="steelblue", label="Residuals") # Added label for legend
ax.axvline(0, color="k", linestyle="--")
ax.set_xlabel("Residual (Predicted - True) [K]")
ax.set_ylabel("Count")
ax.set_title(f"Symbolic KAN residuals (mean = {residuals.mean():.2f})");ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import torch
import kan
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score

# ======================================
# SEED
# ======================================

SEED = 42

torch.manual_seed(SEED)
torch.set_default_dtype(torch.float64)

# ======================================
# LOAD 6 DATASETS
# ======================================

r1_gt = pd.read_csv("r1_gt_imu.csv")
r2_gt = pd.read_csv("r2_gt_imu.csv")
r3_gt = pd.read_csv("r3_gt_imu.csv")

imu_cols = [
    'timestamp',
    'wx', 'wy', 'wz',
    'ax', 'ay', 'az'
]

r1_imu = pd.read_csv("r1_imu.txt", sep=r"\s+", comment="#", names=imu_cols)
r2_imu = pd.read_csv("r2_imu.txt", sep=r"\s+", comment="#", names=imu_cols)
r3_imu = pd.read_csv("r3_imu.txt", sep=r"\s+", comment="#", names=imu_cols)

# combine all GT datasets

gt_data = pd.concat(
    [r1_gt, r2_gt, r3_gt],
    ignore_index=True
)

imu_data = pd.concat(
    [r1_imu, r2_imu, r3_imu],
    ignore_index=True
)

# ONLY if row counts match
data = pd.concat(
    [gt_data.reset_index(drop=True),
     imu_data.reset_index(drop=True)],
    axis=1
)

print("Total Samples:", len(data))

# ======================================
# QUATERNION TO EULER
# ======================================

def quaternion_to_euler(qx, qy, qz, qw):

    # Roll
    sinr_cosp = 2 * (qw * qx + qy * qz)
    cosr_cosp = 1 - 2 * (qx*qx + qy*qy)

    roll = np.arctan2(sinr_cosp, cosr_cosp)

    # Pitch
    sinp = 2 * (qw*qy - qz*qx)

    pitch = np.where(
        np.abs(sinp) >= 1,
        np.sign(sinp) * np.pi / 2,
        np.arcsin(sinp)
    )

    # Yaw
    siny_cosp = 2 * (qw*qz + qx*qy)
    cosy_cosp = 1 - 2 * (qy*qy + qz*qz)

    yaw = np.arctan2(siny_cosp, cosy_cosp)

    return roll, pitch, yaw

# ======================================
# COMPUTE ROLL PITCH YAW
# ======================================

roll, pitch, yaw = quaternion_to_euler(
    data['qx'].values,
    data['qy'].values,
    data['qz'].values,
    data['qw'].values
)

# convert radian to degree

roll = np.degrees(roll)
pitch = np.degrees(pitch)
yaw = np.degrees(yaw)

# ======================================
# INPUT FEATURES
# ======================================

# Corrected feature names to use available columns from the data
feature_names = [
    'tx',
    'ty',
    'tz',
    'wx',
    'wy',
    'wz',
    'ax',
    'ay',
    'az'
]

X_np = data[feature_names].values

# ======================================
# OUTPUTS
# ======================================

y_np = np.column_stack((roll, pitch, yaw))

# ======================================
# HANDLE NaNs
# ======================================
# Combine X_np and y_np to drop NaNs consistently
combined_data_df = pd.DataFrame(X_np, columns=feature_names)
combined_data_df['roll'] = y_np[:, 0]
combined_data_df['pitch'] = y_np[:, 1]
combined_data_df['yaw'] = y_np[:, 2]

# Drop rows with any NaN values
initial_samples = len(combined_data_df)
combined_data_df.dropna(inplace=True)
final_samples = len(combined_data_df)
print(f"Dropped {initial_samples - final_samples} rows containing NaN values.")

# Separate back into X_np and y_np
X_np = combined_data_df[feature_names].values
y_np = combined_data_df[['roll', 'pitch', 'yaw']].values


print("Input Shape:", X_np.shape)
print("Output Shape:", y_np.shape)

# ======================================
# TRAIN TEST SPLIT
# ======================================

X_train_np, X_test_np, y_train_np, y_test_np = train_test_split(
    X_np,
    y_np,
    test_size=0.2,
    random_state=SEED
)

# ======================================
# NORMALIZATION
# ======================================

scaler_x = MinMaxScaler()
scaler_y = MinMaxScaler()

X_train = torch.tensor(
    scaler_x.fit_transform(X_train_np)
)

X_test = torch.tensor(
    scaler_x.transform(X_test_np)
)

y_train = torch.tensor(
    scaler_y.fit_transform(y_train_np)
)

y_test = torch.tensor(
    scaler_y.transform(y_test_np)
)

# ======================================
# DATASET FOR KAN
# ======================================

dataset = {
    'train_input': X_train,
    'train_label': y_train,
    'test_input': X_test,
    'test_label': y_test
}

# ======================================
# CREATE KAN MODEL
# ======================================

# Changed width to match the new number of input features (3 for tx, ty, tz)
model = kan.KAN(
    width=[9, 10, 3],
    grid=5,
    k=3,
    seed=SEED
)

# ======================================
# TRAIN
# ======================================

model.fit(
    dataset,
    opt="LBFGS",
    steps=100,
    lamb=1e-4
)

# ======================================
# PRUNE
# ======================================

model = model.prune()

model.fit(
    dataset,
    opt="LBFGS",
    steps=50
)

# ======================================
# PREDICTION
# ======================================

with torch.no_grad():

    y_pred_scaled = model(X_test)

# ======================================
# INVERSE TRANSFORM
# ======================================

y_pred = scaler_y.inverse_transform(
    y_pred_scaled.numpy()
)

y_true = scaler_y.inverse_transform(
    y_test.numpy()
)

# ======================================
# R2 SCORE
# ======================================

r2_roll = r2_score(y_true[:,0], y_pred[:,0])
r2_pitch = r2_score(y_true[:,1], y_pred[:,1])
r2_yaw = r2_score(y_true[:,2], y_pred[:,2])

print("\n===== R2 SCORES =====")

print(f"Roll  R2 : {r2_roll:.4f}")
print(f"Pitch R2 : {r2_pitch:.4f}")
print(f"Yaw   R2 : {r2_yaw:.4f}")

# ======================================
# PLOTS
# ======================================

labels = ['Roll', 'Pitch', 'Yaw']

fig, axes = plt.subplots(1, 3, figsize=(18,5))

for i in range(3):

    ax = axes[i]

    ax.scatter(
        y_true[:,i],
        y_pred[:,i],
        alpha=0.5
    )

    lo = min(y_true[:,i].min(), y_pred[:,i].min())
    hi = max(y_true[:,i].max(), y_pred[:,i].max())

    ax.plot([lo,hi],[lo,hi],'r--')

    ax.set_xlabel("True")
    ax.set_ylabel("Predicted")

    ax.set_title(labels[i])

plt.tight_layout()
plt.show()